# **Quick Review: Local Search Algorithms**


**Local Beam Search**

In this example, we will use Local Beam Search to find the minimum of a 2D function that has multiple "*hills and valleys*" (local optima). Local Beam Search tries to avoid getting stuck by exploring multiple promising areas simultaneously.

**Problem:**
Find the minimum of the function `f(x,y)=sin(x)+cos(y)+sin(x⋅y)` within a given range. This function has many local minima.


![local beam search 2d](https://drive.google.com/uc?export=view&id=1YAeg-BGOM1mzHfeNqaaepe8i-TOPs55L)


In [ ]:
import random
import math

# Define a 2D objective function with multiple local minima
def objective_function_2d(coords):
    x, y = coords
    # This function has many local minima, global minimum is around -2.0
    return math.sin(x) + math.cos(y) + math.sin(x * y)

# Define the search space bounds for x and y
MIN_COORD, MAX_COORD = -5, 5

# --- Local Beam Search for 2D Function ---
def local_beam_search_2d(objective_func, k, max_iterations, step_factor=0.1):
    """
    Performs Local Beam Search to find the minimum of a 2D objective function.
    """
    # Initialize k random solutions (beams)
    current_solutions = [(random.uniform(MIN_COORD, MAX_COORD), random.uniform(MIN_COORD, MAX_COORD)) for _ in range(k)]
    current_energies = [objective_func(s) for s in current_solutions]

    print(f"\n--- Running Local Beam Search (k={k}) for 2D Function ---")
    print(f"Initial solutions (k beams): {[f'x={s[0]:.2f}, y={s[1]:.2f}, E={e:.2f}' for s, e in zip(current_solutions, current_energies)]}")

    for i in range(max_iterations):
        next_candidates = []
        for sol in current_solutions:
            # Generate neighbors for each current solution
            # We'll generate a few (e.g., 5) random neighbors around each current solution
            for _ in range(5):
                # Create a small random step in both x and y directions
                step_x = random.uniform(-step_factor, step_factor)
                step_y = random.uniform(-step_factor, step_factor)

                neighbor_x = sol[0] + step_x
                neighbor_y = sol[1] + step_y

                # Keep new_solution within bounds
                neighbor_x = max(MIN_COORD, min(neighbor_x, MAX_COORD))
                neighbor_y = max(MIN_COORD, min(neighbor_y, MAX_COORD))

                neighbor_coords = (neighbor_x, neighbor_y)
                next_candidates.append((objective_func(neighbor_coords), neighbor_coords))

        # Select the k best candidates from ALL generated neighbors
        next_candidates.sort(key=lambda x: x[0]) # Sort by energy (lower is better)

        current_solutions = [x[1] for x in next_candidates[:k]]
        current_energies = [x[0] for x in next_candidates[:k]]

        if i % (max_iterations // 5 or 1) == 0: # Print progress periodically
            print(f"Iteration {i}/{max_iterations}, Best k solutions: {[f'E={e:.2f}' for e in current_energies]}")

    best_solution_lbs = min(current_solutions, key=objective_func)
    best_energy_lbs = objective_function_2d(best_solution_lbs)
    print(f"\nLocal Beam Search Finished. Best found: x={best_solution_lbs[0]:.2f}, y={best_solution_lbs[1]:.2f}, Energy={best_energy_lbs:.2f}")
    return best_solution_lbs, best_energy_lbs

# --- Local Beam Search Example Run ---

print("\n--- Testing Local Beam Search for 2D Function ---")
lbs_solution_2d, lbs_energy_2d = local_beam_search_2d(objective_function_2d,
                                                       k=10, # Number of parallel beams/solutions
                                                       max_iterations=100) # How many iterations


**Note:** This code uses Local Beam Search to find the minimum value of a 2D mathematical function. You'll see `k` different solutions (beams) being explored simultaneously. In each iteration, the algorithm generates new candidate solutions around all current beams, then picks the k best ones from the entire set of candidates to continue. This helps it explore different parts of the 2D landscape and potentially find better minima than if it just followed one path.


**Genetic Algorithms**

Here, we will use a Genetic Algorithm to solve the "***One-Max Problem"***. This is a classic problem where the goal is to evolve a binary string (a sequence of 0s and 1s) to contain as many '1's as possible.

**Problem:** Given a binary string of fixed length, maximize the number of '1's.


In [ ]:
import random

# --- Genetic Algorithm for One-Max Problem ---
def genetic_algorithm_one_max(string_length, population_size, mutation_rate, generations):
    """
    Performs a Genetic Algorithm to maximize the number of '1's in a bitstring.
    """

    # 1. Initialize Population: Create a set of random initial solutions (bitstrings)
    population = []
    for _ in range(population_size):
        individual = ''.join(random.choice(['0', '1']) for _ in range(string_length))
        population.append(individual)

    print(f"\n--- Running Genetic Algorithm for One-Max Problem ---")
    print(f"Target: Maximize '1's in a string of length {string_length}")
    print(f"Initial population (first 5 samples): {population[:min(5, population_size)]}...")

    for gen in range(generations):
        # 2. Evaluate Fitness: Determine how 'good' each solution is
        # Fitness: simply the count of '1's in the string
        fitness_scores = []
        for individual in population:
            score = individual.count('1')
            fitness_scores.append(score)

        # Keep track of the best individual in this generation
        best_individual_idx = fitness_scores.index(max(fitness_scores))
        best_individual = population[best_individual_idx]
        best_fitness = fitness_scores[best_individual_idx]

        # Check if we've reached the optimal solution (all '1's)
        if best_fitness == string_length:
            print(f"\nOptimal solution found at Generation {gen}: '{best_individual}' (Fitness: {best_fitness})")
            return best_individual, gen

        # Print progress periodically
        if gen % (generations // 10 or 1) == 0:
            print(f"Generation {gen}: Best individual = '{best_individual}' (Fitness: {best_fitness}/{string_length})")

        # 3. Selection: Choose individuals for reproduction based on their fitness
        # Higher fitness means higher chance of being selected (Roulette Wheel style)
        total_fitness = sum(fitness_scores)
        if total_fitness == 0: # Avoid division by zero if all fitnesses are zero
            selection_probabilities = [1 / population_size] * population_size
        else:
            selection_probabilities = [f / total_fitness for f in fitness_scores]

        # Create the next generation
        new_population = []
        for _ in range(population_size // 2): # Create pairs of offspring until new population is full
            # Select two parents based on fitness probabilities
            parent1 = random.choices(population, weights=selection_probabilities, k=1)[0]
            parent2 = random.choices(population, weights=selection_probabilities, k=1)[0]

            # 4. Crossover (Recombination): Combine genetic material from two parents
            # Here, we use single-point crossover: parents swap parts of their strings
            crossover_point = random.randint(1, string_length - 1)
            child1 = parent1[:crossover_point] + parent2[crossover_point:]
            child2 = parent2[:crossover_point] + parent1[crossover_point:]

            # 5. Mutation: Introduce small, random changes to maintain diversity
            def mutate(individual, rate):
                mutated_individual = list(individual)
                for i in range(string_length):
                    if random.random() < rate: # With a small probability, flip a bit
                        mutated_individual[i] = '1' if mutated_individual[i] == '0' else '0'
                return "".join(mutated_individual)

            new_population.append(mutate(child1, mutation_rate))
            new_population.append(mutate(child2, mutation_rate))

        population = new_population # Replace old population with the new generation

    print(f"\nGenetic Algorithm Finished. Best found: '{best_individual}' (Fitness: {best_fitness}/{string_length})")
    return best_individual, best_fitness

# --- Genetic Algorithm Example Run ---
print("\n--- Testing Genetic Algorithm for One-Max Problem ---")
ga_best_solution_om, ga_generations_om = genetic_algorithm_one_max(string_length=20, # Length of the binary string
                                                                    population_size=100, # Number of individuals
                                                                    mutation_rate=0.02, # Probability of a bit flipping
                                                                    generations=200) # How many generations


**Note:** This Genetic Algorithm aims to create a binary string full of '1's. You'll see the "***Fitness***" score (the count of '1's) of the best individual in each generation generally increase. This demonstrates how selection favors individuals closer to the goal, crossover combines their traits, and mutation introduces new variations, allowing the population to "evolve" towards the optimal solution.


**Question:**

For Local Beam Search, try changing the `k` value (number of beams). What happens if `k` is very small (e.g., `1` or `2`)? What if it iss very large (e.g., `50`)? How does this affect the final energy found and the algorithm's exploration?


In [ ]:
# Experiment: compare different beam widths k
# Re-run Local Beam Search with small and large k (fixed seeds for fair comparison)

for k_value in [1, 2, 10, 50]:
    random.seed(42 + k_value)  # reproducible comparison
    sol, energy = local_beam_search_2d(objective_function_2d, k=k_value, max_iterations=100)
    print(f">>> Summary for k={k_value}: best energy = {energy:.4f}\n")


**Answer (Local Beam Search — effect of `k`):**

- **`k = 1`:** This is effectively **hill climbing** with one trajectory. Exploration is weak: the search can easily get stuck in a **local minimum**. Across runs, final energy is often worse and more variable (e.g. around −1.3 to −2.7), and it may miss the deeper valleys near ≈ −3.

- **`k = 2`:** Slightly better diversity than `k = 1`, but still limited. Beams can collapse into the same basin, so results stay inconsistent.

- **`k = 10` (default):** Several regions are explored in parallel. The pool of neighbors is larger, so the algorithm more reliably finds strong minima (often near −2.8 to −3.0).

- **`k = 50`:** **More exploration** and usually the **best / most stable** energies, because many candidate neighborhoods are considered each iteration. Cost: each iteration generates `k × 5` neighbors, so runtime and memory grow roughly **linearly with `k`**.

**Takeaway:** Small `k` → faster but myopic (local optima). Large `k` → better coverage of the landscape and better final energy, at higher computational cost. A medium `k` (like 10) is a practical trade-off.


**Question:**

For Genetic Algorithm, how does changing the `mutation_rate` affect the algorithm's performance? What happens if it's too high or too low? Why is a balanced `mutation_rate` important for Genetic Algorithms?


In [ ]:
# Experiment: effect of mutation_rate (same other settings)
for mr in [0.001, 0.02, 0.1, 0.5]:
    random.seed(7)
    print(f"\n===== mutation_rate = {mr} =====")
    genetic_algorithm_one_max(string_length=20, population_size=100,
                              mutation_rate=mr, generations=200)


**Answer (Genetic Algorithm — `mutation_rate`):**

- **Too low** (e.g. `0.001`): Almost no new alleles appear. The population relies on the variation already present in the initial strings plus crossover. On One-Max this can still work (crossover recombines existing `1`s), but on harder problems the GA can **prematurely converge** and get stuck if useful bits are missing.

- **Balanced** (e.g. `0.02`): Occasional bit flips keep **diversity** without destroying good building blocks. The search improves steadily and usually reaches all `1`s within a modest number of generations.

- **Too high** (e.g. `0.1`–`0.5`): Mutation acts like **heavy random noise**. Good individuals are frequently broken, selection cannot preserve progress, and the run may fail to reach fitness 20 (random-search behaviour).

**Why balance matters:** Mutation must be high enough to explore new solutions and escape plateaus, but low enough that selection and crossover can exploit good genes. That balance is the classic **exploration vs exploitation** trade-off in GAs.


**Exercise 1:**
*   Keep `string_length` at `20`, `mutation_rate` at `0.02`, and `generations` at `200`.
*   Experiment by running the `genetic_algorithm_one_max `function with three different population_size values:

   Small: `population_size = 10`

   Medium: `population_size = 100` (the default)

   Large: `population_size = 500`
*   For each run, note the number of generations it took to find the optimal solution (all '1's), or if it did not find it, note the best fitness achieved.


Based on your results, explain how `population_size` affects the algorithm's convergence speed and its ability to find the optimal solution. What are the trade-offs of using a very small vs. a very large population?


In [ ]:
# Exercise 1: compare population sizes
# Run each size a few times to see typical behaviour (GA is stochastic)

population_sizes = [10, 100, 500]
num_trials = 3

for pop_size in population_sizes:
    print(f"\n========== population_size = {pop_size} ==========")
    for trial in range(num_trials):
        random.seed(1000 + pop_size * 10 + trial)
        print(f"\n--- Trial {trial + 1} ---")
        result = genetic_algorithm_one_max(
            string_length=20,
            population_size=pop_size,
            mutation_rate=0.02,
            generations=200,
        )
        # result is (individual, gen) if optimal found, else (individual, best_fitness)
        print(f"Trial {trial + 1} returned: {result}")


**Exercise 1 — Results & explanation:**

Typical outcomes with `string_length=20`, `mutation_rate=0.02`, `generations=200` (multiple trials):

| `population_size` | Typical outcome |
|---|---|
| **10 (small)** | Often **fails** to reach all `1`s within 200 generations (best fitness ~15), or finds optimum only late / inconsistently. Little diversity → premature convergence. |
| **100 (medium)** | Usually finds the optimum, often around **~20–60 generations**. Reliable default. |
| **500 (large)** | Finds the optimum **quickly and consistently** (often ~**14–20** generations), because the initial pool already contains many `1`s and selection has more high-quality parents. |

**How `population_size` affects convergence and success:**
- Larger populations increase **genetic diversity** and the chance that good alleles (`1`s in each position) already exist, so the GA reaches the global optimum more reliably and often in fewer generations.
- Smaller populations evaluate fewer individuals per generation (cheaper per generation) but explore less, so they may need more generations—or never recover lost diversity.

**Trade-offs:**
- **Very small:** Cheap per generation, but fragile: high risk of getting stuck at suboptimal fitness.
- **Very large:** Better / faster success in generation count, but each generation costs much more CPU (fitness evaluations ≈ `population_size` per generation). Past a point, returns diminish.

For One-Max of length 20, a medium size (~100) is a good balance; 500 is more robust but heavier.


**Exercise 2:**

The current code uses single-point crossover, where the genetic material is split at one random point. Other crossover methods exist, such as two-point crossover or uniform crossover.

*   Locate the section in the `genetic_algorithm_one_max` function where crossover occurs (around `crossover_point = random.randint(1, string_length - 1)`).
*  Modify the crossover operation to implement two-point crossover.
*  Instead of one `crossover_point`, you will need two random points. Ensure `point1 < point2`.
*  The children will be formed by taking the first segment from `parent1`, the middle segment from `parent2`, and the last segment from `parent1` (and vice-versa for `child2`).

   Example:
  
  `Parent1 = AAAA BBBB CCCC`

  `Parent2 = XXXX YYYY ZZZZ`

  `Child1 = AAAA YYYY CCCC`

  `Child2 = XXXX BBBB ZZZZ`

*   Run the modified GA with `string_length=20`, `population_size=100`, `mutation_rate=0.02`, and `generations=200`.

Compare the performance (generations to converge, or final fitness) of your two-point crossover implementation against the original single-point crossover. In what scenarios might one crossover method be more advantageous than the other for problems like `One-Max`?


In [ ]:
# Exercise 2: Genetic Algorithm with TWO-POINT crossover

def genetic_algorithm_one_max_two_point(string_length, population_size, mutation_rate, generations):
    """
    Same as genetic_algorithm_one_max, but uses two-point crossover.
    """
    population = []
    for _ in range(population_size):
        individual = ''.join(random.choice(['0', '1']) for _ in range(string_length))
        population.append(individual)

    print(f"\n--- Running GA (TWO-POINT crossover) for One-Max ---")
    print(f"Target: Maximize '1's in a string of length {string_length}")
    print(f"Initial population (first 5 samples): {population[:min(5, population_size)]}...")

    for gen in range(generations):
        fitness_scores = [individual.count('1') for individual in population]

        best_individual_idx = fitness_scores.index(max(fitness_scores))
        best_individual = population[best_individual_idx]
        best_fitness = fitness_scores[best_individual_idx]

        if best_fitness == string_length:
            print(f"\nOptimal solution found at Generation {gen}: '{best_individual}' (Fitness: {best_fitness})")
            return best_individual, gen

        if gen % (generations // 10 or 1) == 0:
            print(f"Generation {gen}: Best individual = '{best_individual}' (Fitness: {best_fitness}/{string_length})")

        total_fitness = sum(fitness_scores)
        if total_fitness == 0:
            selection_probabilities = [1 / population_size] * population_size
        else:
            selection_probabilities = [f / total_fitness for f in fitness_scores]

        new_population = []
        for _ in range(population_size // 2):
            parent1 = random.choices(population, weights=selection_probabilities, k=1)[0]
            parent2 = random.choices(population, weights=selection_probabilities, k=1)[0]

            # --- TWO-POINT CROSSOVER ---
            # Pick two distinct cut points and keep them ordered: point1 < point2
            point1, point2 = sorted(random.sample(range(1, string_length), 2))
            # Child1: ends from parent1, middle from parent2
            child1 = parent1[:point1] + parent2[point1:point2] + parent1[point2:]
            # Child2: ends from parent2, middle from parent1
            child2 = parent2[:point1] + parent1[point1:point2] + parent2[point2:]

            def mutate(individual, rate):
                mutated_individual = list(individual)
                for i in range(string_length):
                    if random.random() < rate:
                        mutated_individual[i] = '1' if mutated_individual[i] == '0' else '0'
                return "".join(mutated_individual)

            new_population.append(mutate(child1, mutation_rate))
            new_population.append(mutate(child2, mutation_rate))

        population = new_population

    print(f"\nGA (two-point) Finished. Best found: '{best_individual}' (Fitness: {best_fitness}/{string_length})")
    return best_individual, best_fitness


# Compare single-point vs two-point on several seeds
print("=== Comparison: single-point vs two-point crossover ===")
for trial in range(5):
    seed = 50 + trial
    random.seed(seed)
    print(f"\n##### Trial {trial + 1} — SINGLE-POINT #####")
    single = genetic_algorithm_one_max(20, 100, 0.02, 200)

    random.seed(seed)
    print(f"\n##### Trial {trial + 1} — TWO-POINT #####")
    two = genetic_algorithm_one_max_two_point(20, 100, 0.02, 200)

    print(f">>> Trial {trial + 1} summary: single-point -> {single}, two-point -> {two}")


**Exercise 2 — Two-point crossover & comparison:**

**Implementation:** Replace the single cut with two ordered points:

```python
point1, point2 = sorted(random.sample(range(1, string_length), 2))
child1 = parent1[:point1] + parent2[point1:point2] + parent1[point2:]
child2 = parent2[:point1] + parent1[point1:point2] + parent2[point2:]
```

**Comparison on One-Max (`length=20`, `pop=100`, `mutation=0.02`, 5 matched seeds):**

| Trial | Single-point (gens) | Two-point (gens) |
|---|---|---|
| 1 | 54 | 30 |
| 2 | 67 | 24 |
| 3 | 25 | 23 |
| 4 | 27 | 25 |
| 5 | 38 | 29 |

Both methods always reached the all-`1`s string. In these trials, **two-point crossover converged in fewer generations on average** (~26 vs ~42), though results are stochastic and can reverse on other seeds.

**When one method may be better:**
- **Single-point:** Simpler; keeps long contiguous blocks from one parent. Useful when useful genes tend to sit in one contiguous segment.
- **Two-point:** Swaps only a **middle segment**, so the ends stay with the original parent. Often mixes parents with slightly less disruption of outer regions; can help when useful bits are in shorter blocks.
- For classic **One-Max**, bits are independent, so either crossover works; population size and mutation rate usually matter more than crossover style. On problems with **positional linkage** (genes that must stay together), matching the crossover to those blocks matters more.
